In [17]:
# Los tres laberintos representados como listas de listas

# Definición de las grillas de prueba.
# "S" representa el punto de partida.
# "G" representa la meta.
# "." representa un camino transitable.
# "#" representa un obstáculo o pared.


laberinto_con_solucion = [
    ["S", ".", ".", "#", ".", "."],
    ["#", "#", ".", "#", ".", "#"],
    [".", ".", ".", ".", ".", "."],
    [".", "#", "#", "#", "#", "."],
    [".", ".", ".", "#", ".", "."],
    ["#", "#", ".", ".", ".", "G"]
]

# La columna 2 es un muro vertical completo: S y G quedan en lados opuestos
laberinto_sin_solucion = [
    ["S", ".", "#", ".", ".", "."],
    ["#", ".", "#", ".", "#", "."],
    [".", ".", "#", ".", ".", "."],
    [".", ".", "#", ".", ".", "."],
    [".", ".", "#", ".", ".", "."],
    ["#", ".", "#", ".", ".", "G"]
]

laberinto_varios_caminos = [
    ["S", ".", ".", ".", ".", "."],
    [".", "#", "#", "#", ".", "."],
    [".", ".", ".", "#", ".", "."],
    [".", "#", ".", "#", "#", "."],
    [".", ".", ".", ".", ".", "#"],
    ["#", "#", ".", ".", ".", "G"]
]

In [18]:
# Funciones de apoyo para trabajar con el laberinto

def imprimir_laberinto(laberinto):
    # Imprime el laberinto fila por fila
    for fila in laberinto:
        print(" ".join(fila))

def copiar_laberinto(laberinto):
    # Crea una copia fila por fila para no modificar el original
    copia = []
    for fila in laberinto:
        copia.append(fila[:])
    return copia

def buscar_posicion(laberinto, simbolo):
    # Recorre el laberinto buscando el símbolo dado (S o G)
    for i in range(len(laberinto)):
        for j in range(len(laberinto[0])):
            if laberinto[i][j] == simbolo:
                return (i, j)
    return None

def obtener_vecinos(posicion):
    # Devuelve las 4 celdas vecinas: arriba, abajo, izquierda, derecha
    fila, columna = posicion
    vecinos = []
    vecinos.append((fila, columna + 1))  # derecha
    vecinos.append((fila + 1, columna))  # abajo
    vecinos.append((fila, columna - 1))  # izquierda
    vecinos.append((fila - 1, columna))  # arriba
    return vecinos

def esta_dentro(laberinto, posicion):
    # Verifica que la posición esté dentro de los límites del laberinto
    fila, columna = posicion
    return 0 <= fila < len(laberinto) and 0 <= columna < len(laberinto[0])

def es_valido(laberinto, posicion, visitados):
    # Un movimiento es válido si: está dentro, no es pared y no fue visitado
    if not esta_dentro(laberinto, posicion):
        return False
    fila, columna = posicion
    if laberinto[fila][columna] == "#":
        return False
    if posicion in visitados:
        return False
    return True

# Probamos las funciones con el primer laberinto
print("Laberinto con solución:")
imprimir_laberinto(laberinto_con_solucion)
print("Entrada:", buscar_posicion(laberinto_con_solucion, "S"))
print("Salida: ", buscar_posicion(laberinto_con_solucion, "G"))

Laberinto con solución:
S . . # . .
# # . # . #
. . . . . .
. # # # # .
. . . # . .
# # . . . G
Entrada: (0, 0)
Salida:  (5, 5)


In [19]:
# Algoritmo principal: backtracking para resolver el laberinto

def resolver_laberinto(laberinto):
    entrada = buscar_posicion(laberinto, "S")
    salida  = buscar_posicion(laberinto, "G")

    if entrada is None or salida is None:
        raise ValueError("El laberinto debe tener una entrada S y una salida G.")

    # El camino empieza en la entrada; visitados evita repetir celdas (poda)
    camino    = [entrada]
    visitados = {entrada}

    estadisticas = {
        "llamadas_recursivas": 0,
        "ramas_descartadas":   0,
        "retrocesos":          0
    }

    def backtracking(posicion_actual):
        estadisticas["llamadas_recursivas"] += 1

        # Condición de solución completa: llegamos a la salida
        if posicion_actual == salida:
            return True

        # Probamos cada vecino como candidato
        for vecino in obtener_vecinos(posicion_actual):

            # Restricciones: descartamos candidatos no válidos
            if not es_valido(laberinto, vecino, visitados):
                estadisticas["ramas_descartadas"] += 1
                continue

            # Tomamos la decisión: avanzamos al vecino
            camino.append(vecino)
            visitados.add(vecino)

            if backtracking(vecino):
                return True

            # Retroceso: el vecino no llevó a la salida, deshacemos el paso
            camino.pop()
            visitados.remove(vecino)
            estadisticas["retrocesos"] += 1

        return False

    encontrado = backtracking(entrada)
    return encontrado, camino, estadisticas

In [20]:
# Función para mostrar el camino encontrado marcando las celdas con *

def laberinto_con_camino(laberinto, camino):
    nuevo = copiar_laberinto(laberinto)
    for fila, columna in camino:
        # Marcamos con * solo las celdas intermedias (no S ni G)
        if nuevo[fila][columna] not in ("S", "G"):
            nuevo[fila][columna] = "*"
    return nuevo


# Probamos el primer laberinto de forma individual
encontrado, camino, stats = resolver_laberinto(laberinto_con_solucion)

print("¿Se encontró camino?", encontrado)
print("Camino:", camino)
print("Estadísticas:", stats)
print("\nSolución:")
imprimir_laberinto(laberinto_con_camino(laberinto_con_solucion, camino))

¿Se encontró camino? True
Camino: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (3, 5), (4, 5), (5, 5)]
Estadísticas: {'llamadas_recursivas': 11, 'ramas_descartadas': 5, 'retrocesos': 0}

Solución:
S * * # . .
# # * # . #
. . * * * *
. # # # # *
. . . # . *
# # . . . G


In [21]:
# Probamos los tres laberintos en secuencia

casos = [
    ("Laberinto con solución",       laberinto_con_solucion),
    ("Laberinto sin solución",        laberinto_sin_solucion),
    ("Laberinto con varios caminos",  laberinto_varios_caminos)
]

for nombre, laberinto in casos:
    print("=" * 60)
    print(nombre)
    print("Laberinto inicial:")
    imprimir_laberinto(laberinto)

    encontrado, camino, stats = resolver_laberinto(laberinto)

    print("\n¿Se encontró camino?", encontrado)
    print("Estadísticas:", stats)

    if encontrado:
        print("Camino encontrado:")
        imprimir_laberinto(laberinto_con_camino(laberinto, camino))
    else:
        print("No existe camino bajo las condiciones dadas.")

    print()

Laberinto con solución
Laberinto inicial:
S . . # . .
# # . # . #
. . . . . .
. # # # # .
. . . # . .
# # . . . G

¿Se encontró camino? True
Estadísticas: {'llamadas_recursivas': 11, 'ramas_descartadas': 5, 'retrocesos': 0}
Camino encontrado:
S * * # . .
# # * # . #
. . * * * *
. # # # # *
. . . # . *
# # . . . G

Laberinto sin solución
Laberinto inicial:
S . # . . .
# . # . # .
. . # . . .
. . # . . .
. . # . . .
# . # . . G

¿Se encontró camino? False
Estadísticas: {'llamadas_recursivas': 25, 'ramas_descartadas': 76, 'retrocesos': 24}
No existe camino bajo las condiciones dadas.

Laberinto con varios caminos
Laberinto inicial:
S . . . . .
. # # # . .
. . . # . .
. # . # # .
. . . . . #
# # . . . G

¿Se encontró camino? True
Estadísticas: {'llamadas_recursivas': 36, 'ramas_descartadas': 80, 'retrocesos': 25}
Camino encontrado:
S . . . . .
* # # # . .
* * * # . .
. # * # # .
. . * * * #
# # . . * G

